In [77]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import roc_auc_score,accuracy_score, classification_report , confusion_matrix

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

In [40]:
RANDOM_STATE = 42

In [41]:
df=pd.read_csv(r'data/train.csv')

In [42]:

df=df.drop(columns=['id'])

In [31]:
df

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,30,71090.0,27.4,2,5,6,5.0,Male,Urban,Sedan,No,Yes,Medium,No
668661,32,129990.0,5.0,2,5,4,1.0,Female,Suburban,SUV,Yes,Yes,Low,No
668662,64,121791.0,24.2,1,5,6,1.0,Female,Suburban,Hatchback,Yes,Yes,Low,No
668663,51,115923.0,43.7,2,0,0,1.0,Female,Rural,SUV,Yes,Yes,Low,No


In [43]:
df['Will_Buy_EV']=df['Will_Buy_EV'].map({'Yes':1,'No':0})

In [44]:
df

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,0
1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,0
2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,1
3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,0
4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,30,71090.0,27.4,2,5,6,5.0,Male,Urban,Sedan,No,Yes,Medium,0
668661,32,129990.0,5.0,2,5,4,1.0,Female,Suburban,SUV,Yes,Yes,Low,0
668662,64,121791.0,24.2,1,5,6,1.0,Female,Suburban,Hatchback,Yes,Yes,Low,0
668663,51,115923.0,43.7,2,0,0,1.0,Female,Rural,SUV,Yes,Yes,Low,0


In [45]:
binary_cols=['Home_Charging_Possible','Subsidy_Available']

for cols in binary_cols:
    df[cols]=df[cols].map({'Yes':1,'No':0})
df

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,1,0,Low,0
1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,1,0,Low,0
2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,0,1,Low,1
3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,1,0,Low,0
4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,1,0,Low,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,30,71090.0,27.4,2,5,6,5.0,Male,Urban,Sedan,0,1,Medium,0
668661,32,129990.0,5.0,2,5,4,1.0,Female,Suburban,SUV,1,1,Low,0
668662,64,121791.0,24.2,1,5,6,1.0,Female,Suburban,Hatchback,1,1,Low,0
668663,51,115923.0,43.7,2,0,0,1.0,Female,Rural,SUV,1,1,Low,0


In [46]:
rangeAnxiety={"Low":0,"Medium":1,"High":2}

df['Range_Anxiety_Level']=df['Range_Anxiety_Level'].map(rangeAnxiety)
df

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,1,0,0,0
1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,1,0,0,0
2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,0,1,0,1
3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,1,0,0,0
4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,30,71090.0,27.4,2,5,6,5.0,Male,Urban,Sedan,0,1,1,0
668661,32,129990.0,5.0,2,5,4,1.0,Female,Suburban,SUV,1,1,0,0
668662,64,121791.0,24.2,1,5,6,1.0,Female,Suburban,Hatchback,1,1,0,0
668663,51,115923.0,43.7,2,0,0,1.0,Female,Rural,SUV,1,1,0,0


In [47]:
nominal_cols = ["Gender", "City_Type", "Current_Car_Type"]
df = pd.get_dummies(df, columns=nominal_cols, drop_first=False)

In [48]:
df

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,...,Gender_Female,Gender_Male,Gender_Other,City_Type_Rural,City_Type_Suburban,City_Type_Urban,Current_Car_Type_Hatchback,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck
0,66,92887.0,23.4,2,3,7,1.0,1,0,0,...,False,True,False,False,True,False,False,False,True,False
1,38,30000.0,5.0,1,2,2,4.0,1,0,0,...,False,True,False,True,False,False,False,True,False,False
2,26,94389.0,36.8,1,8,15,5.0,0,1,0,...,True,False,False,False,False,True,False,False,True,False
3,66,73580.0,23.7,2,6,9,3.0,1,0,0,...,False,True,False,False,True,False,True,False,False,False
4,54,57898.0,50.8,1,2,3,3.0,1,0,0,...,False,True,False,False,True,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,30,71090.0,27.4,2,5,6,5.0,0,1,1,...,False,True,False,False,False,True,False,False,True,False
668661,32,129990.0,5.0,2,5,4,1.0,1,1,0,...,True,False,False,False,True,False,False,True,False,False
668662,64,121791.0,24.2,1,5,6,1.0,1,1,0,...,True,False,False,False,True,False,True,False,False,False
668663,51,115923.0,43.7,2,0,0,1.0,1,1,0,...,True,False,False,True,False,False,False,True,False,False


In [49]:
X=df.drop(columns=['Will_Buy_EV'])
y=df['Will_Buy_EV']


In [51]:
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.3,stratify=y, random_state=RANDOM_STATE)

In [53]:
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.fit_transform(x_test)

In [54]:
log_reg=LogisticRegression(max_iter=1000)
log_reg.fit(x_train_scaled,y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [56]:
print("Logistic Regression AUC:",
      roc_auc_score(y_test, log_reg.predict_proba(x_test_scaled)[:, 1]))

Logistic Regression AUC: 0.9382179357587974


In [57]:
rf=RandomForestClassifier(n_estimators=300,n_jobs=-1,random_state=RANDOM_STATE)

In [63]:
rf.fit(x_train,y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [64]:
print("random forest ROC , auc :", roc_auc_score(y_test,rf.predict_proba(x_test)[:,1]))

random forest ROC , auc : 0.9347491028560456


In [83]:
xgb_clf=xgb.XGBClassifier(n_estimators=500,learning_rate=0.05,max_depth=6,subsample=0.8,colsample_bytree=0.8,eval_metric='auc',n_jobs=-1,random_state=RANDOM_STATE)

In [84]:
xgb_clf.fit(x_train,y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'auc'


In [85]:
print("XGBoost AUC:",
      roc_auc_score(y_test, xgb_clf.predict_proba(x_test)[:, 1]))

XGBoost AUC: 0.941755926527553


In [86]:
lgb_clf = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE
)

In [87]:
lgb_clf.fit(x_train, y_train)
print("LightGBM AUC:",
      roc_auc_score(y_test, lgb_clf.predict_proba(x_test)[:, 1]))
 

[LightGBM] [Info] Number of positive: 81745, number of negative: 386320
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003573 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 629
[LightGBM] [Info] Number of data points in the train set: 468065, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174645 -> initscore=-1.553061
[LightGBM] [Info] Start training from score -1.553061
LightGBM AUC: 0.9414113927921007


In [88]:
cat_clf = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6,
    verbose=0, random_state=RANDOM_STATE
)
cat_clf.fit(x_train, y_train)
print("CatBoost AUC:",
      roc_auc_score(y_test, cat_clf.predict_proba(x_test)[:, 1]))
 

CatBoost AUC: 0.9413738949422223


In [89]:
voting = VotingClassifier(
    estimators=[("xgb", xgb_clf), ("lgb", lgb_clf), ("cat", cat_clf)],
    voting="soft",
)
voting.fit(x_train, y_train)
print("Voting Ensemble AUC:",
      roc_auc_score(y_test, voting.predict_proba(x_test)[:, 1]))
 

[LightGBM] [Info] Number of positive: 81745, number of negative: 386320
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003861 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 629
[LightGBM] [Info] Number of data points in the train set: 468065, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174645 -> initscore=-1.553061
[LightGBM] [Info] Start training from score -1.553061
Voting Ensemble AUC: 0.9417789434913353


In [91]:
stack = StackingClassifier(
    estimators=[("xgb", xgb_clf), ("lgb", lgb_clf), ("cat", cat_clf)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1,
)
stack.fit(x_train, y_train)
print("Stacking Ensemble AUC:",
      roc_auc_score(y_test, stack.predict_proba(x_test)[:, 1]))
 

Stacking Ensemble AUC: 0.9417761971475965


In [100]:
best_model = stack
y_pred = best_model.predict(x_test)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.8984247258225324

Classification report:
               precision    recall  f1-score   support

           0       0.93      0.95      0.94    165566
           1       0.74      0.65      0.69     35034

    accuracy                           0.90    200600
   macro avg       0.83      0.80      0.82    200600
weighted avg       0.89      0.90      0.90    200600

Confusion matrix:
 [[157386   8180]
 [ 12196  22838]]
